In [ ]:
import numpy as np                # cálculo numérico
import pandas as pd              # manejo de DataFrames

from scipy import signal         # procesamiento de señales
from scipy.stats import (
    shapiro,        # normalidad
    levene,         # homocedasticidad
    ttest_ind,      # prueba t
    mannwhitneyu    # prueba no paramétrica
)

import mne
import os
from scipy.io import loadmat


In [17]:
ruta = r"control\C001R_EP_reposo.mat"
rutaP = r"parkinson\P001_EP_reposo.mat"
mat = loadmat(ruta)
matp = loadmat(rutaP)

print("control: ",mat.keys())
print("parkinson:", matp.keys())

dataControl = mat['data']
dataParkinson = matp['data']
print("shape Control: ", dataControl.shape)
print("shape Parkinson:", dataParkinson.shape)


control:  dict_keys(['__header__', '__version__', '__globals__', 'data'])
parkinson: dict_keys(['__header__', '__version__', '__globals__', 'data'])
shape Control:  (8, 2000, 180)
shape Parkinson: (8, 2000, 143)


Los datos anteriores, permiten tener una visualizacion de como es la estructura de los datos, asi se tienen, 8 canales, 2000 muestras por epoca y 180-143 epocas

##  1. Energia promedio por canal  ##

In [19]:
def energia_promedio_por_canal(data):
    """
    data: array con forma (canales, muestras, epocas)
    
    retorna:
    vector de tamaño (canales,) con la energía promedio por canal
    """
    
    # Energía por época (sumando sobre muestras)
    energia_epoca = np.sum(data**2, axis=1)  
    # shape: (canales, epocas)
    
    # Promedio sobre épocas
    energia_promedio = np.mean(energia_epoca, axis=1)  
    # shape: (canales,)
    
    return energia_promedio

## 2. DataFrame con energia de cada canal para cada sujeto y grupo ## 

In [20]:
def procesar_grupo(ruta_carpeta):
    """
    ruta_carpeta: path a carpeta (control o parkinson)

    retorna:
        df: DataFrame (filas = sujetos, columnas = canales)
    """
    
    energias = []
    nombres = []

    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".mat"):
            ruta = os.path.join(ruta_carpeta, archivo)
            
            mat = loadmat(ruta)
            data = mat['data']  # (8, 2000, n_epocas)

            # Calcular energía por canal
            energia = energia_promedio_por_canal(data)
            
            energias.append(energia)
            nombres.append(archivo)

    # Crear DataFrame
    df = pd.DataFrame(energias)
    
    # Nombrar columnas como canales
    df.columns = [f"Canal_{i+1}" for i in range(df.shape[1])]
    
    # Índice como sujetos
    df.index = nombres

    return df

In [21]:
ruta_control = r"control"
ruta_parkinson = r"parkinson"

df_control = procesar_grupo(ruta_control)
df_parkinson = procesar_grupo(ruta_parkinson)

In [ ]:
print(df_control.shape)
print(df_parkinson.shape)

df_control


(36, 8)
(23, 8)


,Canal_1,Canal_2,Canal_3,Canal_4,Canal_5,Canal_6,Canal_7,Canal_8
C001R_EP_reposo.mat,21465.650358,20985.907912,22760.149588,18505.640284,29730.163026,25244.158073,22781.327587,24658.599512
C002_EP_reposo.mat,15966.402868,17617.810248,20804.937129,19654.400017,16678.982063,93894.049009,66862.496275,75685.125872
C004_EP_reposo.mat,14148.673322,18283.999666,28749.932148,14270.726911,28787.445978,14661.417740,15940.154095,19499.898656
C005_EP_reposo_Repetido.mat,35311.301696,34916.686010,38800.429029,35427.031127,35905.472869,106598.128152,106885.575966,112520.750636
C006_EP_reposo.mat,18510.829979,19738.489375,20911.792748,21828.254399,23351.992649,53086.059766,37495.972475,43067.095504
C007_EP_reposo.mat,13180.109317,13925.217812,16218.994223,12324.883659,14060.300659,25767.024864,21935.759622,22827.781293
C010_EP_reposo.mat,11197.554574,10948.368805,12737.004665,10745.161921,10329.642418,21461.605831,15493.212776,27414.375247
C011_EP_reposo.mat,28551.124065,26204.839254,17383.998956,17244.605933,26206.372422,83370.618408,51121.105722,67852.348256
C012_EP_reposo.mat,9133.036290,9214.155028,11626.411811,10809.621612,10467.879938,46336.873547,45695.801756,50086.361637
C013_EP_reposo.mat,47166.556798,55107.798641,52286.884667,34682.656928,30606.119338,227045.733387,224891.029478,322172.431642


In [25]:
df_parkinson

,Canal_1,Canal_2,Canal_3,Canal_4,Canal_5,Canal_6,Canal_7,Canal_8
P001_EP_reposo.mat,12438.243570,11261.175800,10819.634775,9489.784462,12091.060945,22798.213463,23700.620349,25606.065340
P004_EP_reposo.mat,17995.660058,12001.601821,12286.344400,14785.908284,17058.433161,63983.449318,53715.460772,66403.639479
P005_EP_reposo.mat,38092.102574,43575.379457,41979.994799,41715.287990,46513.737045,251649.394709,179345.438488,262361.180410
P007_EP_reposo.mat,23742.325612,22070.007569,24540.315612,21803.936448,22594.339745,128314.264805,128888.485633,152799.284248
P012_EP_reposo.mat,48574.518921,51806.529769,73171.952374,59707.699631,56552.175747,287105.761622,222745.793414,353312.298104
P013_EP_reposo.mat,16202.416566,13124.247855,13988.674335,12752.027365,15784.724049,50730.233172,50742.237835,73694.061647
P015_EP_reposo.mat,10692.948223,10841.187262,12154.390086,24161.685202,14789.173543,43302.825848,42560.941845,39043.973220
P016_EP_reposo.mat,12157.229828,13398.658526,17668.877657,14841.104693,11297.742247,38701.647608,41828.973929,61328.610990
P017_EP_reposo.mat,9581.810471,14008.572615,9589.230257,9374.085669,8154.941858,28970.848994,40705.897395,36624.858559
P018_EP_reposo.mat,23658.738825,23990.255991,30633.745996,22888.894132,19932.315538,65161.432397,60552.834862,58441.048743


## 3. Diferencia Estadistica entre canales de grupos ##

In [ ]:
def analisis_estadistico(df_control, df_parkinson, alpha=0.05):
    resultados = []

    for canal in df_control.columns:
        
        x = df_control[canal].values
        y = df_parkinson[canal].values

        # 1. Normalidad 
        stat_x, p_x = shapiro(x)
        stat_y, p_y = shapiro(y)

        normal = (p_x > alpha) and (p_y > alpha)

        #  2. Homocedasticidad 
        stat_lev, p_lev = levene(x, y)
        iguales_var = p_lev > alpha

        #  3. Selección de prueba 
        if normal and iguales_var:
            stat, p_val = ttest_ind(x, y)
            test = "t-test"
        
        elif normal and not iguales_var:
            stat, p_val = ttest_ind(x, y, equal_var=False)
            test = "t-test Welch"
        
        else:
            stat, p_val = mannwhitneyu(x, y)
            test = "Mann-Whitney"

        #  4. Resultado 
        significativo = p_val < alpha

        resultados.append({
            "Canal": canal,
            "Normalidad": normal,
            "p_normal_control": p_x,
            "p_normal_parkinson": p_y,
            "Homocedasticidad": iguales_var,
            "p_levene": p_lev,
            "Test": test,
            "p_value": p_val,
            "Diferencia_significativa": significativo
        })

    return pd.DataFrame(resultados)

In [29]:
df_resultados = analisis_estadistico(df_control, df_parkinson)

df_resultados


,Canal,Normalidad,p_normal_control,p_normal_parkinson,Homocedasticidad,p_levene,Test,p_value,Diferencia_significativa
0,Canal_1,False,6.252583e-03,0.014392,True,0.884802,Mann-Whitney,0.405701,False
1,Canal_2,False,3.954350e-03,0.004522,True,0.959093,Mann-Whitney,0.570528,False
2,Canal_3,False,8.902474e-03,0.000910,True,0.977634,Mann-Whitney,0.460373,False
3,Canal_4,False,1.969822e-04,0.000312,True,0.759253,Mann-Whitney,0.234462,False
4,Canal_5,False,7.995109e-04,0.005323,True,0.904555,Mann-Whitney,0.560018,False
5,Canal_6,False,6.077846e-06,0.000010,True,0.471976,Mann-Whitney,0.280075,False
6,Canal_7,False,4.219023e-06,0.000017,True,0.617876,Mann-Whitney,0.118301,False
7,Canal_8,False,5.307788e-07,0.000009,True,0.594077,Mann-Whitney,0.150544,False


In [ ]:
canales_significativos = df_resultados[df_resultados["Diferencia_significativa"]]

print(canales_significativos[["Canal", "Test", "p_value"]])

Empty DataFrame
Columns: [Canal, Test, p_value]
Index: []


In [32]:
df_resultados["media_control"] = df_control.mean().values
df_resultados["media_parkinson"] = df_parkinson.mean().values
df_resultados

,Canal,Normalidad,p_normal_control,p_normal_parkinson,Homocedasticidad,p_levene,Test,p_value,Diferencia_significativa,media_control,media_parkinson
0,Canal_1,False,6.252583e-03,0.014392,True,0.884802,Mann-Whitney,0.405701,False,22014.673247,24349.672173
1,Canal_2,False,3.954350e-03,0.004522,True,0.959093,Mann-Whitney,0.570528,False,23786.918304,25381.112899
2,Canal_3,False,8.902474e-03,0.000910,True,0.977634,Mann-Whitney,0.460373,False,25845.875622,29464.724079
3,Canal_4,False,1.969822e-04,0.000312,True,0.759253,Mann-Whitney,0.234462,False,22883.189648,26006.722152
4,Canal_5,False,7.995109e-04,0.005323,True,0.904555,Mann-Whitney,0.560018,False,23827.649490,25789.030015
5,Canal_6,False,6.077846e-06,0.000010,True,0.471976,Mann-Whitney,0.280075,False,83518.395864,112586.818643
6,Canal_7,False,4.219023e-06,0.000017,True,0.617876,Mann-Whitney,0.118301,False,70348.417360,94258.342250
7,Canal_8,False,5.307788e-07,0.000009,True,0.594077,Mann-Whitney,0.150544,False,92530.994769,123540.589405


## Introducción: ##

El análisis de bioseñales electroencefalográficas (EEG) se ha consolidado como una herramienta clínica no invasiva fundamental para el estudio de trastornos neurológicos como la enfermedad de Parkinson. Diversos estudios han reportado que las alteraciones en la actividad eléctrica cerebral pueden ser cuantificadas mediante características como la energía de la señal, lo que sugiere su posible uso como biomarcador para diferenciar entre pacientes con Parkinson y sujetos sanos [1].

En este contexto, el presente trabajo tiene como objetivo realizar una comparación estadística de la energía promedio de señales EEG multicanal entre un grupo control y un grupo de pacientes con enfermedad de Parkinson, con el fin de identificar posibles diferencias significativas entre ambos grupos y evaluar el potencial discriminativo de esta métrica.

## Analisis de resultados: ##


Los resultados obtenidos muestran que, para todos los canales analizados, no se encontraron diferencias estadísticamente significativas en la energía promedio de las señales EEG entre el grupo control y el grupo de pacientes con enfermedad de Parkinson (p > 0.05). Debido a que los datos no cumplieron el supuesto de normalidad, se utilizó la prueba no paramétrica de Mann-Whitney, la cual es adecuada para este tipo de distribución.

A pesar de la ausencia de significancia estadística, se observa una tendencia consistente en varios canales, especialmente en los canales 6, 7 y 8, donde el grupo de pacientes con Parkinson presenta valores de energía promedio mayores en comparación con el grupo control. Este comportamiento sugiere que podrían existir diferencias en la actividad eléctrica cerebral entre ambos grupos, aunque estas no son lo suficientemente marcadas como para ser detectadas como significativas bajo el enfoque estadístico utilizado.

Estos resultados indican que la energía total de la señal EEG, considerada de forma global, puede no ser una característica suficientemente sensible para discriminar entre sujetos sanos y pacientes con Parkinson en este conjunto de datos.


## Discusion ##

La ausencia de diferencias estadísticamente significativas puede explicarse por varios factores. En primer lugar, las señales EEG son inherentemente variables tanto entre sujetos como dentro de un mismo individuo, lo que puede aumentar la dispersión de los datos y dificultar la detección de diferencias claras entre grupos.

En segundo lugar, aunque la literatura sugiere que la energía de la señal EEG puede ser útil como biomarcador [1], en muchos casos estas diferencias no se manifiestan en la energía total de la señal, sino en componentes específicas del espectro de frecuencias. En particular, estudios han reportado alteraciones en bandas como la beta en pacientes con enfermedad de Parkinson [2], lo que no necesariamente se refleja en una medida global como la energía total.

Otro aspecto relevante es el tamaño de la muestra. Aunque el número de sujetos analizados es adecuado para un estudio exploratorio, podría no ser suficiente para detectar diferencias pequeñas pero clínicamente relevantes. Esto es especialmente importante en bioseñales, donde la variabilidad interindividual puede enmascarar patrones subyacentes.

Adicionalmente, el uso de la energía total como única característica limita la capacidad de análisis, ya que esta métrica no permite distinguir entre diferentes tipos de actividad cerebral. Por ello, un análisis más detallado, como la descomposición en bandas de frecuencia o el uso de otras características podría ofrecer mejores resultados en la discriminación entre grupos.

## Conclusiones ##

*  No se encontraron diferencias estadísticamente significativas en la energía promedio de las señales EEG entre el grupo control y el grupo de pacientes con enfermedad de Parkinson en ninguno de los canales analizados.
*  Aunque se observó una tendencia a mayores valores de energía en el grupo de Parkinson, especialmente en algunos canales, estas diferencias no fueron suficientes para establecer una distinción estadística entre los grupos.
*  La energía total de la señal EEG no resulta ser un descriptor adecuado por sí solo para discriminar entre sujetos sanos y pacientes con enfermedad de Parkinson en este contexto.
*  Para mejorar la capacidad de discriminación, se recomienda emplear características más específicas de la señal, como el análisis por bandas de frecuencia.

## Bibliografia ##

[1] S. Han et al., "Evaluation of Parkinson's disease early diagnosis using single-channel EEG features and auditory cognitive assessment," Frontiers in Neurology, vol. 14, p. 1273458, 2023. https://www.frontiersin.org/journals/neurology/articles/10.3389/fneur.2023.1273458/full 

[2] T. Başar, "Brain oscillations in neuropsychiatric disease," Dialogues in Clinical Neuroscience, vol. 15, no. 3, pp. 291–300, 2013. 

